# try kubakub nodes in your browser

ComfyUI + [kubakub nodes](https://github.com/kubakubkub/ComfyUI_KubaNodes) on a free Colab GPU.

1. **Runtime → Change runtime type → T4 GPU**
2. Run the three cells from top to bottom (**Runtime → Run all**), about 10 minutes with the models.
3. The last cell prints a link: open it, then **Workflow → Browse templates → ComfyUI_KubaNodes**.

Without any model these run at once: **01 regions in 30 seconds**, **03 director playground**, **shader**,
**mask_animate**, **sound**, **region_materials** and the **kubakub timeline** (bottom panel).
With the models (cell 2): **02 one prompt per region**, a facade made region by region with Flux 2 Klein 4B.
Every file field left empty runs a built-in sample facade, so nothing has to be uploaded.

In [ ]:
# 1. ComfyUI 0.43.0 + kubakub nodes 0.7.0 (about 2 minutes)
!nvidia-smi --query-gpu=name,memory.total --format=csv
%cd /content
!git clone -q --depth 1 --branch v0.43.0 https://github.com/comfyanonymous/ComfyUI
%cd /content/ComfyUI
!pip install -q -r requirements.txt
!git clone -q --depth 1 --branch v0.7.0 https://github.com/kubakubkub/ComfyUI_KubaNodes custom_nodes/ComfyUI_KubaNodes
!pip install -q -r custom_nodes/ComfyUI_KubaNodes/requirements.txt
!apt-get -qq install -y ffmpeg > /dev/null

**2. Flux 2 Klein 4B** for workflow 02 (about 10 GB of downloads, 3-8 minutes).
The fp8 files fit Colab's memory; they are saved under the names the workflows load.
Skip this cell to try the nodes without AI models first.

In [ ]:
# 2. models (optional)
%cd /content/ComfyUI/models
!wget -q -nc -O diffusion_models/flux-2-klein-4b.safetensors https://huggingface.co/black-forest-labs/FLUX.2-klein-4b-fp8/resolve/main/flux-2-klein-4b-fp8.safetensors
!wget -q -nc -O text_encoders/qwen_3_4b.safetensors https://huggingface.co/Comfy-Org/z_image_turbo/resolve/main/split_files/text_encoders/qwen_3_4b_fp8_mixed.safetensors
!wget -q -nc -O vae/flux2-vae.safetensors https://huggingface.co/Comfy-Org/flux2-dev/resolve/main/split_files/vae/flux2-vae.safetensors
!ls -la diffusion_models text_encoders vae | grep safetensors
%cd /content/ComfyUI

In [ ]:
# 3. start ComfyUI and open a public link to it (cloudflared quick tunnel, no account)
!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb && dpkg -i cloudflared-linux-amd64.deb > /dev/null
import subprocess, time, re, socket
subprocess.Popen(["python", "main.py", "--listen", "127.0.0.1", "--port", "8188"], cwd="/content/ComfyUI",
                 stdout=open("/content/comfy.log", "w"), stderr=subprocess.STDOUT)
while True:
    try:
        socket.create_connection(("127.0.0.1", 8188), 1).close(); break
    except OSError:
        time.sleep(1)
p = subprocess.Popen(["cloudflared", "tunnel", "--url", "http://127.0.0.1:8188"], stderr=subprocess.PIPE, text=True)
for line in p.stderr:
    url = re.search(r"https://[\w-]+\.trycloudflare\.com", line)
    if url:
        print("open:", url.group(0)); break
print("keep this cell running; the ComfyUI log is in /content/comfy.log")